In [ ]:
import os
import cv2
import time
import numpy as np
from PIL import Image

import torch
from facenet_pytorch import MTCNN, InceptionResnetV1

In [ ]:
KNOWN_DIR = "known_faces"
CAMERA_INDEX = 0
COSINE_THRESHOLD = 0.70
FRAME_DOWNSCALE = 0.75
PROCESS_EVERY_N_FRAMES = 1 
AGGREGATE_PER_PERSON = True

In [ ]:
def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(p=2, dim=-1, keepdim=True) + eps)

def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    a = l2_normalize(a)
    b = l2_normalize(b)
    if a.dim() == 1:
        a = a.unsqueeze(0) 
    return (a @ b.T).squeeze(0)

def pil_from_bgr(frame_bgr: np.ndarray) -> Image.Image:
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

def draw_label(img, x1, y1, x2, y2, text):
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    y = y1 - 10 if y1 - 10 > 10 else y1 + 20
    cv2.putText(img, text, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

In [ ]:
def build_face_db(known_dir: str, mtcnn: MTCNN, resnet: InceptionResnetV1, device: torch.device):

    if not os.path.isdir(known_dir):
        raise FileNotFoundError(f"Dataset folder not found: {known_dir}")

    person_to_embs = {}

    for person in sorted(os.listdir(known_dir)):
        person_path = os.path.join(known_dir, person)
        if not os.path.isdir(person_path):
            continue

        for fname in sorted(os.listdir(person_path)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                continue

            fpath = os.path.join(person_path, fname)
            img = Image.open(fpath).convert("RGB")

            face = mtcnn(img)
            if face is None:
                print(f"[WARN] No face found in {fpath} (skipping)")
                continue

            face = face.unsqueeze(0).to(device)  

            with torch.no_grad():
                emb = resnet(face)  
                emb = l2_normalize(emb).cpu().squeeze(0) 

            person_to_embs.setdefault(person, []).append(emb)
            print(f"[OK] Loaded {person}: {fname}")

    if not person_to_embs:
        raise RuntimeError("No embeddings were created. Check your dataset images.")

    db_names = []
    db_embeddings = []

    if AGGREGATE_PER_PERSON:
        for person, embs in person_to_embs.items():
            stacked = torch.stack(embs, dim=0) 
            mean_emb = l2_normalize(stacked.mean(dim=0))
            db_names.append(person)
            db_embeddings.append(mean_emb)
        db_embeddings = torch.stack(db_embeddings, dim=0)
    else:
        for person, embs in person_to_embs.items():
            for emb in embs:
                db_names.append(person)
                db_embeddings.append(emb)
        db_embeddings = torch.stack(db_embeddings, dim=0) 

    return db_embeddings, db_names

In [ ]:
def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    mtcnn = MTCNN(image_size=160, margin=14, keep_all=True, device=device)

    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    print("Building known face database...")
    db_embeddings, db_names = build_face_db(KNOWN_DIR, mtcnn, resnet, device)
    print(f"Database size: {len(db_names)} identities")

    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam. Try changing CAMERA_INDEX (0,1,2,...)")

    frame_i = 0
    last_time = time.time()

    print("Webcam started. Press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_i += 1
        if PROCESS_EVERY_N_FRAMES > 1 and (frame_i % PROCESS_EVERY_N_FRAMES != 0):
            cv2.imshow("FaceNet Recognition", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
            continue

        if FRAME_DOWNSCALE != 1.0:
            small = cv2.resize(frame, (0, 0), fx=FRAME_DOWNSCALE, fy=FRAME_DOWNSCALE)
        else:
            small = frame

        pil_img = pil_from_bgr(small)

        boxes, probs = mtcnn.detect(pil_img)
        if boxes is not None and len(boxes) > 0:
            faces = mtcnn(pil_img)
            if faces is not None:
                faces = faces.to(device)

                with torch.no_grad():
                    embs = resnet(faces)        
                    embs = l2_normalize(embs).cpu() 

                for idx, (box, prob) in enumerate(zip(boxes, probs)):
                    if prob is None or prob < 0.90:
                        continue 

                    emb = embs[idx] 
                    sims = cosine_similarity(emb, db_embeddings) 
                    best_i = int(torch.argmax(sims).item())
                    best_sim = float(sims[best_i].item())
                    name = db_names[best_i] if best_sim >= COSINE_THRESHOLD else "Unknown"

                    x1, y1, x2, y2 = box
                    if FRAME_DOWNSCALE != 1.0:
                        x1 = int(x1 / FRAME_DOWNSCALE)
                        y1 = int(y1 / FRAME_DOWNSCALE)
                        x2 = int(x2 / FRAME_DOWNSCALE)
                        y2 = int(y2 / FRAME_DOWNSCALE)
                    else:
                        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

                    label = f"{name}  sim={best_sim:.3f}"
                    draw_label(frame, x1, y1, x2, y2, label)

        now = time.time()
        fps = 1.0 / max(now - last_time, 1e-6)
        last_time = now
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.imshow("FaceNet Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
